# Augmentierungsvergleich
Ungefärbt → Gefärbt angleichen für Domain Transfer

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms.functional as TF
import torchvision.transforms as T
from pathlib import Path
import random

## 1. Bilder laden

In [ ]:
# Pfade anpassen
path_ungefaerbt = Path('ungefaerbt.png')
path_gefaerbt   = Path('PE-2025-01953-M_00_s0060_PM_Complete_Transmittance_Stitched_Flat_v000_512_4352.png')
path_maske      = Path('PE-2025-01953-M_00_s0060_PM_Complete_Transmittance_Stitched_Flat_v000_512_4352.png')

img_u = Image.open(path_ungefaerbt).convert('L')
img_g = Image.open(path_gefaerbt).convert('L')
mask  = Image.open(path_maske).convert('L')

# Als Tensor
to_tensor = lambda img: torch.tensor(np.array(img), dtype=torch.float32).unsqueeze(0) / 255.0
t_u = to_tensor(img_u)
t_g = to_tensor(img_g)
t_m = (to_tensor(mask) > 0.5).float()

print(f'Ungefärbt: {t_u.shape}, min={t_u.min():.2f}, max={t_u.max():.2f}')
print(f'Gefärbt:   {t_g.shape}, min={t_g.min():.2f}, max={t_g.max():.2f}')

## 2. Ausgangszustand

In [ ]:
def show(imgs, titles, mask=None):
    n = len(imgs) + (1 if mask is not None else 0)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    for ax, img, title in zip(axes, imgs, titles):
        ax.imshow(img.squeeze().numpy(), cmap='gray', vmin=0, vmax=1)
        ax.set_title(title)
        ax.axis('off')
    if mask is not None:
        axes[-1].imshow(mask.squeeze().numpy(), cmap='Reds', vmin=0, vmax=1)
        axes[-1].set_title('Maske')
        axes[-1].axis('off')
    plt.tight_layout()
    plt.show()

show([t_u, t_g], ['Ungefärbt', 'Gefärbt'], mask=t_m)

## 3. Histogramm-Vergleich

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(t_u.numpy().flatten(), bins=100, color='steelblue', alpha=0.7)
axes[0].set_title('Histogramm: Ungefärbt')
axes[1].hist(t_g.numpy().flatten(), bins=100, color='salmon', alpha=0.7)
axes[1].set_title('Histogramm: Gefärbt')
plt.tight_layout()
plt.show()

## 4. Augmentierungen definieren

In [ ]:
def aug_contrast(img, factor=2.0):
    """Kontrast erhöhen"""
    return TF.adjust_contrast(img, factor)

def aug_brightness(img, factor=0.7):
    """Helligkeit reduzieren (gefärbt ist oft dunkler)"""
    return TF.adjust_brightness(img, factor)

def aug_gamma(img, gamma=1.5):
    """Gamma-Korrektur – dunkelt Mitteltöne ab"""
    return TF.adjust_gamma(img, gamma)

def aug_sharpen(img, factor=3.0):
    """Schärfen – macht Gefäßränder deutlicher"""
    return TF.adjust_sharpness(img, factor)

def aug_clahe(img):
    """CLAHE – lokale Kontrastverbesserung (bestes Mittel für Histologie)"""
    import cv2
    arr = (img.squeeze().numpy() * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    result = clahe.apply(arr)
    return torch.tensor(result, dtype=torch.float32).unsqueeze(0) / 255.0

def aug_equalize(img):
    """Histogramm-Equalisierung"""
    arr = (img.squeeze().numpy() * 255).astype(np.uint8)
    from PIL import ImageOps
    eq = ImageOps.equalize(Image.fromarray(arr))
    return torch.tensor(np.array(eq), dtype=torch.float32).unsqueeze(0) / 255.0

def aug_combined(img):
    """Kombination: CLAHE + Kontrast + Gamma"""
    img = aug_clahe(img)
    img = aug_contrast(img, factor=1.5)
    img = aug_gamma(img, gamma=1.3)
    return img

print('Augmentierungen definiert.')

## 5. Visueller Vergleich aller Augmentierungen

In [ ]:
augmentierungen = {
    'Original (ungefärbt)': t_u,
    'Kontrast x2':          aug_contrast(t_u, 2.0),
    'Helligkeit -30%':      aug_brightness(t_u, 0.7),
    'Gamma 1.5':            aug_gamma(t_u, 1.5),
    'Schärfen x3':          aug_sharpen(t_u, 3.0),
    'CLAHE':                aug_clahe(t_u),
    'Equalisierung':        aug_equalize(t_u),
    'Kombiniert':           aug_combined(t_u),
}

n = len(augmentierungen) + 1
fig, axes = plt.subplots(1, n, figsize=(4*n, 4))

for ax, (title, img) in zip(axes, augmentierungen.items()):
    ax.imshow(img.squeeze().numpy(), cmap='gray', vmin=0, vmax=1)
    ax.set_title(title, fontsize=8)
    ax.axis('off')

axes[-1].imshow(t_g.squeeze().numpy(), cmap='gray', vmin=0, vmax=1)
axes[-1].set_title('Ziel: Gefärbt', fontsize=8, color='red')
axes[-1].axis('off')

plt.suptitle('Augmentierungen vs. Ziel (gefärbt)', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Quantitativer Vergleich (MSE zum Ziel)

In [ ]:
print(f'{'Augmentierung':<25} MSE zum Gefärbten')
print('-' * 45)
for name, img in augmentierungen.items():
    mse = ((img - t_g) ** 2).mean().item()
    print(f'{name:<25} {mse:.4f}')

## 7. Beste Augmentierung mit Maske überlagert

In [ ]:
# Hier die beste Augmentierung eintragen
beste = aug_clahe(t_u)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(axes, [t_u, beste, t_g], ['Ungefärbt', 'Augmentiert (CLAHE)', 'Gefärbt']):
    ax.imshow(img.squeeze().numpy(), cmap='gray', vmin=0, vmax=1)
    # Maske überlagern
    mask_overlay = np.zeros((*img.squeeze().shape, 4))
    mask_overlay[..., 0] = 1.0  # Rot
    mask_overlay[..., 3] = t_m.squeeze().numpy() * 0.4
    ax.imshow(mask_overlay)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()